# AirQuality 2025 extention

- Takes the earliest complete GMT days totaling approximately 7% of valid 2024 observations.
- Copies their values and moves their timestamps to the first period after the original source.
- Adds the required `aqi` column.

In [8]:
from datetime import timedelta
from math import ceil
from pathlib import Path
from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F

FRACTION = 0.07

INPUT_PATH = '/opt/spark/landing/air_quality/hourly_88101_2024.csv'
OUTPUT_FILE = Path('/opt/spark/landing/air_quality/air_quality_2025_update.csv')
TEMP_OUTPUT_PATH = OUTPUT_FILE.with_name('_air_quality_2025_update_tmp')

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('GenerateAirQuality2025')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')

In [2]:
source = (
    spark.read
    .option('header', True)
    .option('inferSchema', False)
    .csv(INPUT_PATH)
    .withColumn('_gmt_ts', F.to_timestamp(F.concat_ws(' ', F.col('Date GMT'), F.col('Time GMT')), 'yyyy-MM-dd HH:mm'))
    .withColumn('_local_ts', F.to_timestamp(F.concat_ws(' ', F.col('Date Local'), F.col('Time Local')), 'yyyy-MM-dd HH:mm'))
    .filter(F.col('_gmt_ts').isNotNull() & F.col('_local_ts').isNotNull())
)

source_stats = source.agg(
    F.count('*').alias('row_count'),
    F.min('_gmt_ts').alias('min_gmt'),
    F.max('_gmt_ts').alias('max_gmt'),
).first()
source_count = source_stats['row_count']
update_count = ceil(source_count * FRACTION)
source_min_gmt = source_stats['min_gmt']
source_max_gmt = source_stats['max_gmt']

print(f'Valid source rows: {source_count:,}')
print(f'7% update rows: {update_count:,}')
print(f'Latest original GMT timestamp: {source_max_gmt}')

Valid source rows: 8,139,551
7% update rows: 569,769
Latest original GMT timestamp: 2025-01-01 09:00:00


In [3]:
daily_counts = (
    source
    .groupBy(F.to_date('_gmt_ts').alias('gmt_date'))
    .count()
    .orderBy('gmt_date')
    .collect()
)

template_rows = 0
last_template_date = None
for day in daily_counts:
    template_rows += day['count']
    last_template_date = day['gmt_date']
    if template_rows >= update_count:
        break

cutoff = last_template_date + timedelta(days=1)
template = (
    source
    .filter(F.col('_gmt_ts') < F.lit(cutoff))
    .persist(StorageLevel.DISK_ONLY)
)

template_min_gmt = source_min_gmt
shift_hours = int(((source_max_gmt + timedelta(hours=1)) - template_min_gmt).total_seconds() / 3600)

print(f'7% target rows: {update_count:,}')
print(f'Rows in complete earliest-day template: {template_rows:,}')
print(f'Template ends on GMT date: {last_template_date}')
print(f'First template GMT timestamp: {template_min_gmt}')
print(f'Timestamp shift: {shift_hours:,} hours')
print(f'First generated GMT timestamp: {source_max_gmt + timedelta(hours=1)}')

7% target rows: 569,769
Rows in complete earliest-day template: 584,516
Template ends on GMT date: 2024-01-27
First template GMT timestamp: 2024-01-01 04:00:00
Timestamp shift: 8,790 hours
First generated GMT timestamp: 2025-01-01 10:00:00


26/09/27 16:14:50 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [4]:
new_gmt_ts = F.expr(f'_gmt_ts + INTERVAL {shift_hours} HOURS')
new_local_ts = F.expr(f'_local_ts + INTERVAL {shift_hours} HOURS')

aqi = F.floor(F.pow(F.rand(seed=42), 3) * 501).cast('int')

generated = (
    template
    .withColumn('Date GMT', F.date_format(new_gmt_ts, 'yyyy-MM-dd'))
    .withColumn('Time GMT', F.date_format(new_gmt_ts, 'HH:mm'))
    .withColumn('Date Local', F.date_format(new_local_ts, 'yyyy-MM-dd'))
    .withColumn('Time Local', F.date_format(new_local_ts, 'HH:mm'))
    .withColumn('Date of Last Change', F.date_format(F.date_add(F.to_date(new_local_ts), 7), 'yyyy-MM-dd'))
    .withColumn('aqi', aqi)
)

output_columns = [column for column in source.columns if not column.startswith('_')] + ['aqi']
generated = generated.select(*output_columns)
generated.show(10, truncate=False)

+----------+-----------+--------+--------------+---+---------+----------+-----+------------------------+----------+----------+----------+--------+------------------+---------------------------+---+-----------+---------+-----------+-----------+---------------------------------------------------------------------+----------+-----------+-------------------+---+
|State Code|County Code|Site Num|Parameter Code|POC|Latitude |Longitude |Datum|Parameter Name          |Date Local|Time Local|Date GMT  |Time GMT|Sample Measurement|Units of Measure           |MDL|Uncertainty|Qualifier|Method Type|Method Code|Method Name                                                          |State Name|County Name|Date of Last Change|aqi|
+----------+-----------+--------+--------------+---+---------+----------+-----+------------------------+----------+----------+----------+--------+------------------+---------------------------+---+-----------+---------+-----------+-----------+-----------------------------------

In [5]:
generated_count = generated.count()
generated_range = generated.select(
    F.min(F.to_timestamp(F.concat_ws(' ', F.col('Date GMT'), F.col('Time GMT')), 'yyyy-MM-dd HH:mm')).alias('min_gmt'),
    F.max(F.to_timestamp(F.concat_ws(' ', F.col('Date GMT'), F.col('Time GMT')), 'yyyy-MM-dd HH:mm')).alias('max_gmt'),
    F.min('aqi').alias('min_aqi'),
    F.max('aqi').alias('max_aqi'),
).first()

print(f'Generated rows: {generated_count:,}')
print(f"Generated GMT range: {generated_range['min_gmt']} through {generated_range['max_gmt']}")
print(f"AQI range: {generated_range['min_aqi']} through {generated_range['max_aqi']}")

Generated rows: 584,516
Generated GMT range: 2025-01-01 10:00:00 through 2025-01-28 05:26:00
AQI range: 0 through 500


In [6]:
# These fields are unquoted in the original source CSV.
UNQUOTED_NUMERIC_COLUMNS = {
    'POC', 'Latitude', 'Longitude', 'Sample Measurement', 'MDL', 'aqi'
}
TEMP_FILE = OUTPUT_FILE.with_suffix('.csv.tmp')

def quoted(value):
    text = str(value).replace('"', '""')
    return f'"{text}"'

def format_value(column, value):
    text = '' if value is None else str(value)
    return text if column in UNQUOTED_NUMERIC_COLUMNS else quoted(text)

def write_generated_csv(dataframe, columns, destination):
    with destination.open('w', encoding='utf-8', newline='') as stream:
        stream.write(','.join(quoted(column) for column in columns) + '\n')

        for row in dataframe.toLocalIterator():
            row_values = row.asDict(recursive=True)
            csv_row = [format_value(column, row_values[column]) for column in columns]
            stream.write(','.join(csv_row) + '\n')

write_generated_csv(generated, output_columns, TEMP_FILE)
TEMP_FILE.replace(OUTPUT_FILE)
print(f'Wrote the update to: {OUTPUT_FILE}')

Wrote the update to: /opt/spark/landing/air_quality/air_quality_2025_update.csv


In [9]:
spark.stop()